In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import numpy as np

sys.path.append(str(Path.cwd().parent))

In [ ]:
import matplotlib.pyplot as plt
from src.data.pipeline import make_loader
from src.data.splits import load_config, INTERIM_DIR
from src.data.labels import LABEL_MAP_PATH
loader = make_loader("val", 224, 16)
images, labels = next(iter(loader))

categories = list(load_config(LABEL_MAP_PATH))

fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for k, ax in enumerate(axes.flat):
    # 1. ax.imshow(...) of image k, gray colormap   (hint: images[k, 0] -> (224, 224))
    ax.imshow(images[k,0], cmap="gray")
    # 2. title: the category names where labels[k] == 1   (empty -> "none of the 14")
    names = [cat for cat, val in zip(categories, labels[k]) if val==1]
    title = "\n".join(names) if names else "none of the 14"
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()


In [ ]:
from src.models.encoders import SmallCNN
import torch
model = SmallCNN([16, 32, 64, 128, 256], 14)
print(model)
model.features(torch.zeros(2, 1, 224, 224)).shape, sum(p.numel() for p in model.parameters())


In [ ]:
from src.data.pipeline import make_loader

images, labels = next(iter(make_loader("val", 224, 16)))
logits = model(images)
logits.shape, torch.sigmoid(logits).min().item(), torch.sigmoid(logits).max().item()


In [ ]:
loader = next(iter(make_loader("val", 224, 16)))


In [ ]:
from src.utils import set_seeds
import torch
from torch import nn
set_seeds(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
images, labels = next(iter(make_loader("val", 224, 16)))
images, labels = images.to(device), labels.to(device)

model = SmallCNN([16, 32, 64, 128, 256], 14).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model.train()

for step in range(101):
    optimizer.zero_grad()
    logits = model(images)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(step, loss.item())


In [ ]:
torch.sigmoid(model(images))[0].round(decimals=2)

In [ ]:
labels[0]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist = pd.read_csv("../experiments/history/small_cnn_20261110-095132.csv")
plt.plot(hist["epoch"], hist["train_loss"], label="train")
plt.plot(hist["epoch"], hist["val_loss"], label="val")
plt.axhline(0.2059, linestyle="--", color="gray", label="know nothing (val)")
plt.axvline(11, linestyle=":", color="green", label="kept epoch")
plt.ylim(0.10, 0.30)  # without this, the 0.93 spike flattens everything else
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()


In [ ]:
from pathlib import Path
import pandas as pd
from src.utils import append_result

p = Path("toy_results.csv")
append_result({"a": 1, "b": 2}, p)
append_result({"a": 3, "b": 4, "best_epoch": 12}, p)
print(pd.read_csv(p))
p.unlink()          # deletes the toy file
